# Week 1 — OpenCV with CUDA on Google Colab

> *Google Colab üzerinde CUDA destekli OpenCV kurulumu (donanımınız destekliyorsa yerel ortamda VS Code ile de yapılabilir).*

**Why build from source?** The `opencv-python` / `opencv-contrib-python` wheels on PyPI (and the ones Colab pre-installs) are compiled **without CUDA**, so `cv2.cuda.getCudaEnabledDeviceCount()` returns `0` even on a GPU runtime. The GPU modules (`cudaarithm`, `cudafilters`, `cudaimgproc`, …) live in the separate **opencv_contrib** repository and must be compiled with NVIDIA's `nvcc`.

**Before you run:** `Runtime → Change runtime type → T4 GPU`.

What this notebook does:

1. Checks the GPU and its *compute capability* (T4 = 7.5).
2. Optionally mounts Google Drive — if a previous build is cached there, it is restored in seconds.
3. Otherwise clones OpenCV + opencv_contrib and builds **only the modules we need**, for **only this GPU's architecture** (this is what keeps the build to tens of minutes instead of hours).
4. Installs the result into `/opt/opencv-cuda`, hides Colab's CPU-only wheels, and verifies `cv2.cuda` works.


## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
!nvcc --version | tail -n 2

In [ ]:
import subprocess, sys, os

OPENCV_VERSION = "5.0.0"   # same tag for opencv and opencv_contrib
PREFIX = "/opt/opencv-cuda"

try:
    cc = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).split()[0]
except Exception as e:
    raise SystemExit("No NVIDIA GPU found. Runtime → Change runtime type → T4 GPU, then run again.") from e

PYVER = f"{sys.version_info.major}.{sys.version_info.minor}"
print("Compute capability:", cc, "| Python", PYVER, "| OpenCV tag", OPENCV_VERSION)

## 2. (Optional) Cache on Google Drive

The build is slow; the result is not. Set `USE_DRIVE = True` to save the finished build as a `.tar.gz` in your Drive. Next time (new runtime, same GPU type, same Python version) this notebook restores it instead of compiling. The cache name includes the OpenCV version, GPU architecture and Python version, so a mismatch simply triggers a fresh build.

In [ ]:
USE_DRIVE = False

CACHE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/opencv_cuda_cache", exist_ok=True)
    CACHE = f"/content/drive/MyDrive/opencv_cuda_cache/opencv-{OPENCV_VERSION}-sm{cc.replace('.', '')}-py{PYVER}.tar.gz"
    print("Cache file:", CACHE, "(exists)" if os.path.exists(CACHE) else "(not built yet)")

## 3. Build (or restore)

Key CMake options:

| Option | Why |
| --- | --- |
| `WITH_CUDA=ON` | compile the CUDA code paths |
| `OPENCV_EXTRA_MODULES_PATH=…/opencv_contrib/modules` | the `cuda*` modules live in contrib |
| `BUILD_LIST=core,imgproc,imgcodecs,python3,cudev,cudaarithm,cudafilters,cudaimgproc` | build only what we use → much faster |
| `CUDA_ARCH_BIN=<cc>` | generate GPU code only for this card (T4 → `7.5`) |
| `WITH_CUDNN=OFF`, `OPENCV_DNN_CUDA=OFF` | the DNN module is not needed this week |
| `OPENCV_PYTHON3_INSTALL_PATH` | put the `cv2` package inside our own prefix so it can be cached |

The build log is written to `/content/opencv_build.log`; only the tail is printed. Tested on a free Colab T4 runtime (2 CPU cores, CUDA 13.0, Python 3.13): **43 minutes**. The CUDA filter modules take most of that time.

In [ ]:
import shutil, time, multiprocessing

def sh(cmd, log=None):
    print("$", cmd if len(cmd) < 160 else cmd[:157] + "...")
    if log:
        cmd = f"({cmd}) >> {log} 2>&1"
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode != 0:
        if log:
            subprocess.run(f"tail -n 60 {log}", shell=True)
        raise RuntimeError(f"command failed ({r.returncode})")

if CACHE and os.path.exists(CACHE):
    sh(f"mkdir -p {PREFIX} && tar -xzf '{CACHE}' -C {PREFIX}")
    print("Restored from Drive cache.")
else:
    t0 = time.time()
    LOG = "/content/opencv_build.log"
    open(LOG, "w").close()
    sh("apt-get -qq update && apt-get -qq install -y ninja-build", LOG)
    sh("pip -q install --upgrade cmake", LOG)
    os.chdir("/content")
    for repo in ("opencv", "opencv_contrib"):
        if not os.path.isdir(repo):
            sh(f"git clone -q --depth 1 --branch {OPENCV_VERSION} https://github.com/opencv/{repo}.git", LOG)

    import numpy
    cmake = f'''cmake -S /content/opencv -B /content/opencv/build -G Ninja \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX={PREFIX} \
      -DOPENCV_EXTRA_MODULES_PATH=/content/opencv_contrib/modules \
      -DBUILD_LIST=core,imgproc,imgcodecs,python3,cudev,cudaarithm,cudafilters,cudaimgproc \
      -DWITH_CUDA=ON -DCUDA_ARCH_BIN={cc} -DCUDA_ARCH_PTX= \
      -DWITH_CUDNN=OFF -DOPENCV_DNN_CUDA=OFF -DCUDA_FAST_MATH=ON -DWITH_CUBLAS=ON \
      -DBUILD_TESTS=OFF -DBUILD_PERF_TESTS=OFF -DBUILD_EXAMPLES=OFF -DBUILD_DOCS=OFF \
      -DBUILD_opencv_apps=OFF -DBUILD_JAVA=OFF -DWITH_QT=OFF -DWITH_GTK=OFF \
      -DPYTHON3_EXECUTABLE={sys.executable} \
      -DPYTHON3_NUMPY_INCLUDE_DIRS={numpy.get_include()} \
      -DOPENCV_PYTHON3_INSTALL_PATH={PREFIX}/python'''
    sh(cmake, LOG)
    sh(f"cmake --build /content/opencv/build -j{multiprocessing.cpu_count()}", LOG)
    sh("cmake --install /content/opencv/build", LOG)
    print(f"Built in {(time.time() - t0) / 60:.1f} min")
    if CACHE:
        sh(f"tar -czf '{CACHE}' -C {PREFIX} .")
        print("Saved cache to Drive:", CACHE)

## 4. Make Python use the CUDA build

Colab ships several CPU-only OpenCV wheels. If they stay installed, `import cv2` may load one of them instead of ours, so we uninstall them and register our build with a `.pth` file (a one-line text file that adds a folder to Python's import path).

The `.pth` file is lost when the runtime is recycled, so **run this cell (and the cache restore above) once per new runtime**. After this cell, use **Runtime → Restart session** only if `cv2` was already imported earlier in the session.

In [ ]:
sh("pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless || true")
import site
pth = os.path.join(site.getsitepackages()[0], "opencv_cuda.pth")
with open(pth, "w") as f:
    f.write(f"{PREFIX}/python\n")
print("wrote", pth)

## 5. Verify

Colab may print `ERROR:root:Error disabling cv2.imshow()` when `cv2` is imported. This is harmless: our build has no GUI module (`highgui`), so there is no `imshow` for Colab to patch. Show images with matplotlib instead.

In [ ]:
import site
site.addsitedir(f"{PREFIX}/python")   # same effect as the .pth file, for this session
import cv2

print("cv2 file   :", cv2.__file__)
print("version    :", cv2.__version__)
print("CUDA devices:", cv2.cuda.getCudaEnabledDeviceCount())
cv2.cuda.printShortCudaDeviceInfo(cv2.cuda.getDevice())

info = cv2.getBuildInformation()
start = info.find("NVIDIA CUDA")
print(info[start:start + 400])
assert cv2.cuda.getCudaEnabledDeviceCount() > 0, "OpenCV was not built with CUDA or no GPU is visible" 

### Quick smoke test: upload → filter → download

A GPU workflow always has the same shape: copy the image to GPU memory (`GpuMat.upload`), run filters there, copy the result back (`download`). The copies cost time, so the GPU pays off for large images or long chains of operations — not for a single small filter.

In [ ]:
import numpy as np
img = (np.random.default_rng(0).random((1080, 1920)) * 255).astype(np.float32)
g = cv2.cuda.GpuMat(); g.upload(img)
blur = cv2.cuda.createGaussianFilter(cv2.CV_32FC1, cv2.CV_32FC1, (5, 5), 1.0)
out = blur.apply(g).download()
ref = cv2.GaussianBlur(img, (5, 5), 1.0)
print("max |GPU - CPU| =", float(np.abs(out - ref).max()))

## Doing this locally instead (VS Code)

Same idea on your own NVIDIA machine: install the NVIDIA driver + CUDA Toolkit (`nvcc`), then run the same `cmake` command with your GPU's compute capability (`nvidia-smi --query-gpu=compute_cap --format=csv`) and an install prefix of your choice. On Windows use the *x64 Native Tools* prompt with Visual Studio's generator instead of Ninja. Then select that Python interpreter in VS Code. Without an NVIDIA GPU there is nothing to accelerate — the Week 4 filter notebooks fall back to the CPU automatically.

Used later by: **`week4/01_prewitt_x_gauss.ipynb`** and **`week4/02_prewitt_y_gauss.ipynb`** (set `USE_DRIVE = True` here so they can restore this build in their own runtime).